In [1]:
from pathlib import Path
import pandas as pd

file_path = 'G:\\My Drive\\KM Jada Files'
upload_path = Path(file_path) / "Uploads"
raw_data_path = Path(file_path) / "raw data"

In [2]:
# Load data from exported CSV files

df_HH = pd.read_csv(upload_path / '01_transform_households.csv')
df_contacts = pd.read_csv(upload_path / '02_Transform_Person_Accounts.csv')
df_acr = pd.read_csv(upload_path / '03_Transform_ACR_Relationships.csv')

In [ ]:
import pandas as pd

# 1. Isolate orphaned contacts (Contacts missing from your successful ACR export)
linked_contact_ids = set(df_acr['FinServ__SourceSystemId__c_Contact'])
df_orphans = df_contacts[~df_contacts['FinServ__SourceSystemId__c'].isin(linked_contact_ids)].copy()

# 2. Build synthetic composite keys for exact matching
df_orphans['match_key'] = (
    df_orphans['LastName'].str.strip().str.lower() #+ '_' + 
    #df_orphans['BillingPostalCode'].str[:5].fillna('')
)

# Extract last name from Household Name (assumes 'LastName Household' format)
df_HH['clean_hh_name'] = df_HH['Name'].str.replace('Household', '', case=False).str.strip()
df_HH['match_key'] = (
    df_HH['clean_hh_name'].str.split().str[-1].str.lower() #+ '_' + 
    #df_HH['BillingPostalCode'].str[:5].fillna('')
)

# 3. Inner merge to generate missing high-confidence ACRs
exact_matches = df_orphans.merge(
    df_HH[['FinServ__SourceSystemId__c', 'match_key', 'Name']], 
    on='match_key', 
    how='inner',
    suffixes=('_Contact', '_HH')
)

# 4. Format for Data Loader
df_auto_acr = exact_matches.rename(columns={
    'FinServ__SourceSystemId__c_Contact': 'FinServ__SourceSystemId__c_Contact',
    'FinServ__SourceSystemId__c_HH': 'FinServ__SourceSystemId__c_HH'
})
df_auto_acr['Roles'] = 'Client'
df_auto_acr['FinServ__PrimaryGroup__c'] = True

In [7]:
df_export = df_auto_acr[['FinServ__SourceSystemId__c_Contact','FinServ__SourceSystemId__c_HH']]
df_export.to_csv(upload_path / '03_Transform_ACR_Relationships_ReTransform.csv', index=False)

In [8]:
from simple_salesforce import Salesforce
import keyring
sf_user = "kenny.myers@financialintegrators.com"
sf_pass = keyring.get_password("salesforce_prod", sf_user)
sf_token = keyring.get_password("salesforce_prod_token", sf_user)

sf = Salesforce(username=sf_user, password=sf_pass, security_token=sf_token)

In [9]:
# Pulls data from Salesforce using SOQL, then applies in-memory filters with Pandas to refine the dataset according to specific business rules.
# This cell is for all account numbers

from numpy import record
import pandas as pd
from simple_salesforce import Salesforce

# Assuming your 'sf' connection object is already active via keyring

# ---------------------------------------------------------
# STEP 1: THE MACRO-FILTER (Server-Side SOQL)
# ---------------------------------------------------------
# Notice how we translate the JSON's "FK_" syntax to standard Salesforce 
# custom relationship syntax ("__r" instead of "__c") for the join.

sf_soql = """
    SELECT 
    AccountId,
    Id
    FROM Contact
"""

print("Executing SOQL Query...")
results = sf.query_all(sf_soql)

# ---------------------------------------------------------
# STEP 2: DATA NORMALIZATION
# ---------------------------------------------------------
# Because we are pulling cross-object fields (FinServ__FinancialAccount__c.Name, FinServ__r.etc), 
# we MUST use json_normalize to flatten the nested dictionaries.
print(f"Fetched {results['totalSize']} total records. Flattening JSON...")

df_raw = pd.json_normalize(results['records']).drop(columns=['attributes'], errors='ignore')

# Clean up the column names to remove the nested dictionary prefixes for easier reading
# Instantly replaces all dots with underscores across all columns
df_raw.columns = df_raw.columns.str.replace('.', '_', regex=False)

# Drop the nested attributes columns created by json_normalize
salesforce_raw = df_raw[df_raw.columns.drop(list(df_raw.filter(regex='attributes')))]

print(f"Final DataFrame shape: {salesforce_raw.shape[0]} rows, {salesforce_raw.shape[1]} columns.")
display(salesforce_raw.head())

Executing SOQL Query...
Fetched 46618 total records. Flattening JSON...
Final DataFrame shape: 46618 rows, 2 columns.


,AccountId,Id
0,0014X00002X3wrdQAB,0034X000033Gn2fQAC
1,0014X00002X3xabQAB,0034X000033Gnq9QAC
2,0015e00000Ol7XKAAZ,0034X000033GnsKQAS
3,0014X00002X403ZQAR,0034X000033GqfiQAC
4,0014X00002X403zQAB,0034X000033GqftQAC
